# URBANopt CLI Docker Smoke Test

This notebook builds a Docker image with URBANopt CLI 1.2.0 and runs quick test commands.
It also demonstrates a Python helper that mounts local `esbe_2026` into the container.

In [ ]:
from pathlib import Path
import subprocess

REPO_ROOT = Path.cwd()
DOCKERFILE = REPO_ROOT / "docker" / "urbanopt-cli" / "Dockerfile"
IMAGE_TAG = "urbanopt-cli:1.2-ubuntu22"

print("Repo root:", REPO_ROOT)
print("Dockerfile:", DOCKERFILE)
print("Image tag:", IMAGE_TAG)
subprocess.run(["docker", "--version"], check=False)

In [ ]:
# Build the Docker image
build_cmd = [
    "docker",
    "build",
    "-t",
    IMAGE_TAG,
    "-f",
    str(DOCKERFILE),
    str(REPO_ROOT),
]
print("Building Docker image with command:", " ".join(build_cmd))
subprocess.run(build_cmd, check=True)

In [ ]:
# Run basic URBANopt CLI smoke tests
subprocess.run(["docker", "run", "--rm", IMAGE_TAG, "uo", "--version"], check=True)
subprocess.run(["docker", "run", "--rm", IMAGE_TAG, "uo", "--help"], check=True)

In [ ]:
# Use the helper script to run against local esbe_2026 files
from tools.urbanopt_docker import UrbanoptDockerRunner

runner = UrbanoptDockerRunner(image_tag=IMAGE_TAG, mount_subdir="esbe_2026")
result = runner.run_uo(["--help"])
print("Exit code:", result.returncode)
print(result.stdout[:1200])
if result.stderr:
    print("stderr:", result.stderr[:1200])

In [ ]:
# Example shell command from inside mounted esbe_2026
ls_result = runner.run_shell("pwd && ls -la | head -n 30")
print("Exit code:", ls_result.returncode)
print(ls_result.stdout)
if ls_result.stderr:
    print("stderr:", ls_result.stderr)